# imports

In [73]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, accuracy_score
import mlflow
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import mlflow.tensorflow
from tensorflow.keras.callbacks import EarlyStopping

# config

In [74]:
ASSET = "BTC"
INTERVAL = "1h"

In [75]:
TIMESTEPS = 24

# mlflow

In [76]:
mlflow.set_tracking_uri("http://localhost:5000")

experiment_name = f"{ASSET}_{INTERVAL}_deep_learning"
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='/mlruns/13', creation_time=1778414378831, experiment_id='13', last_update_time=1778414378831, lifecycle_stage='active', name='BTC_1h_deep_learning', tags={}, trace_location=None, workspace='default'>

# load data

In [77]:
train_df = pd.read_parquet('../../../data/processed/train_btc_1h.parquet')
test_df = pd.read_parquet('../../../data/processed/test_btc_1h.parquet')

# splitting

In [78]:
features = [col for col in train_df.columns if col not in ['target_1h', 'target_direction']]
X_train = train_df[features]
y_train = train_df['target_direction']
X_test = test_df[features]
y_test = test_df['target_direction']

# checking the train data if scaled or not

In [79]:
print(X_train.describe().loc[['mean', 'std']].T.head(5))

                 mean       std
rsi_14   5.919112e-16  1.000012
roc_10   9.638083e-18  1.000012
roc_20  -2.127025e-17  1.000012
stoch_k -4.254050e-16  1.000012
stoch_d  1.661738e-17  1.000012


# LSTM

In [80]:
def create_sequences(X, y, time_steps=24):
    """
    Transforms 2D tabular data into 3D tensors for LSTM consumption.
    X: Numpy array of features
    y: Target series
    time_steps: historical periods to look back
    """
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:(i + time_steps)])
        ys.append(y.iloc[i + time_steps])
        
    return np.array(Xs), np.array(ys)


X_train_seq, y_train_seq = create_sequences(X_train.values, y_train, TIMESTEPS)
X_test_seq, y_test_seq = create_sequences(X_test.values, y_test, TIMESTEPS)

print(f"Training Shape (Samples, TimeSteps, Features): {X_train_seq.shape}")
print(f"Testing Shape: {X_test_seq.shape}")

Training Shape (Samples, TimeSteps, Features): (42735, 24, 29)
Testing Shape: (10666, 24, 29)


# model

In [81]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(TIMESTEPS, X_train_seq.shape[2])),
    Dropout(0.4),
    
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    
    Dense(16, activation='relu'), Dropout(0.2,),
    
    Dense(1, activation='sigmoid')
])

c:\Users\Manindra\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


# compile model

In [82]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [83]:
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_10 (LSTM)                  │ (None, 24, 64)         │        24,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_11 (LSTM)                  │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 37,025 (144.63 KB)

 Trainable params: 37,025 (144.63 KB)

 Non-trainable params: 0 (0.00 B)

# mlflow tensorflow 

In [84]:
mlflow.tensorflow.autolog(log_models=True, log_input_examples=True)

2026/05/10 15:37:40 WARNING mlflow.utils.autologging_utils: MLflow tensorflow autologging is known to be compatible with 2.16.2 <= tensorflow, but the installed version is 2.16.1. If you encounter errors during autologging, try upgrading / downgrading tensorflow to a compatible version, or try upgrading MLflow.


# early stopping

In [85]:
early_stopping = EarlyStopping(
    monitor='val_loss', 
    patience=15,
    restore_best_weights=True
)

# run, fit model with mlflow

In [86]:
with mlflow.start_run(run_name="LSTM_Tuned_run1"):
    
    mlflow.log_param("time_steps", TIMESTEPS)
    mlflow.log_param("model_type", "LSTM_2_Layer")
    
    print("started training")
    
    history = model.fit(
        X_train_seq, y_train_seq,
        epochs=50,
        batch_size=64,
        validation_split=0.2,
        callbacks=[early_stopping],
        verbose=1
    )
    
    test_loss, test_acc = model.evaluate(X_test_seq, y_test_seq, verbose=0)
    
    mlflow.log_metric("test_loss", test_loss)
    mlflow.log_metric("test_accuracy", test_acc)
    mlflow.keras.log_model(model, "lstm_model")
    
    print(f"\n--- Training Complete ---")
    print(f"Test Accuracy: {test_acc:.4f}")

started training


Epoch 1/50
532/535 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5013 - loss: 0.6968

535/535 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - accuracy: 0.5013 - loss: 0.6967 - val_accuracy: 0.5133 - val_loss: 0.6924
Epoch 2/50
535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5127 - loss: 0.6933 - val_accuracy: 0.5121 - val_loss: 0.6928
Epoch 3/50
535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5111 - loss: 0.6926 - val_accuracy: 0.5098 - val_loss: 0.6929
Epoch 4/50
535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5090 - loss: 0.6929 - val_accuracy: 0.5111 - val_loss: 0.6927
Epoch 5/50
535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5098 - loss: 0.6931 - val_accuracy: 0.5109 - val_loss: 0.6926
Epoch 6/50
535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5136 - loss: 0.6924 - val_accuracy: 0.5153 - val_loss: 0.6926
Epoch 7/50
535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5144 - loss: 0.6923 - val_accuracy: 0.5126 - val_loss: 0.6927
Epoch 8/50
535/535 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.5151 - loss: 0.6921 - val_accuracy: 0.51

2026/05/10 15:39:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 15:40:05 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Manindra\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\saving\saving_lib.py:415: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 12 variables whereas the saved optimizer has 22 variables. "


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 261ms/step


2026/05/10 15:40:05 INFO mlflow.models.model: Found the following environment variables used during model inference: [BYBIT_API_KEY, POSTHOG_API_KEY]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
2026/05/10 15:40:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 15:40:08 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.



--- Training Complete ---
Test Accuracy: 0.5074
🏃 View run LSTM_Tuned_run1 at: http://localhost:5000/#/experiments/13/runs/c4e01a600b10407586b24eab2c1cce26
🧪 View experiment at: http://localhost:5000/#/experiments/13
